In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
import optuna

scaler = MinMaxScaler()
df_courses_tasks = pd.read_csv("Submission/train/courses_tasks_train.csv")
df_activity_log = pd.read_csv("Submission/train/activity_log_train.csv")
df_students = pd.read_csv("Submission/train/students_train.csv")
df_task_marks = pd.read_csv("Submission/train/task_marks_train.csv")
df_labels = pd.read_csv("Submission/train/final_marks_train.csv")

In [2]:
# NEW - Breadth and intensity of study/engagement a student has with non graded tasks.
def course_task_non_grade_revised(df_courses_tasks, df_activity_log):
    """
    Computes student-level features based on engagement with non-graded tasks,
    aggregated per course to maintain context.
    - non_graded_interactions_per_course: intensity of study
    - unique_non_graded_tasks_viewed_per_course: breadth of study
    """
    non_graded_tasks = df_courses_tasks[
        (df_courses_tasks["is_resource"] == True) | (df_courses_tasks["weight"] == 0)
    ]

    logs_non_graded = df_activity_log[
        df_activity_log["task_id"].isin(non_graded_tasks["task_id"])
    ].copy()

    if logs_non_graded.empty:
        return pd.DataFrame(columns=[
            "student_id",
            "course_id",
            "non_graded_interactions_per_course",
            "unique_non_graded_tasks_viewed_per_course"
        ])

    activity_counts_per_course = (
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .count()
        .reset_index(name="non_graded_interactions_per_course")
    )

    task_coverage_per_course = (
        logs_non_graded.groupby(["student_id", "course_id"])["task_id"]
        .nunique()
        .reset_index(name="unique_non_graded_tasks_viewed_per_course")
    )

    features_per_course = activity_counts_per_course.merge(
        task_coverage_per_course, on=["student_id", "course_id"], how="outer"
    )

    return features_per_course.fillna(0)


# Base training frame: unique (student_id, course_id) pairs from labels.
df_train_base = df_labels[['student_id', 'course_id']].drop_duplicates()

# Generate the non-graded features.
student_course_non_graded_features = course_task_non_grade_revised(df_courses_tasks, df_activity_log)

# Merge student-level features (df_students is the actual loaded frame).
df_train = df_train_base.merge(df_students, on='student_id', how='left')

# Merge the per-course non-graded features.
df_train = df_train.merge(
    student_course_non_graded_features,
    on=['student_id', 'course_id'],
    how='left'
)

df_train[['non_graded_interactions_per_course', 'unique_non_graded_tasks_viewed_per_course']] = df_train[
    ['non_graded_interactions_per_course', 'unique_non_graded_tasks_viewed_per_course']
].fillna(0)


In [3]:
# NEW - Course-level difficulty
def compute_difficulty_revised(df_labels):
    """
    Computes course difficulty based on the average final mark for each course.
    Lower average mark indicates a higher difficulty.
    
    Args:
        df_labels (pd.DataFrame): DataFrame with 'course_id' and 'final_mark' for each student.
        
    Returns:
        pd.DataFrame: A DataFrame with 'course_id' and 'course_difficulty'.
    """
    # 1. Group by course and compute the average final mark for each course.
    course_avg_mark = df_labels.groupby("course_id")["final_mark"].mean().reset_index(name="avg_final_mark")
    
    # 2. Compute course difficulty as the inverse of the normalized average final mark.
    # We use a simple normalization to scale the difficulty from 0 to 1.
    # The lowest average mark will get a difficulty of 1, and the highest will get a difficulty of 0.
    min_mark = course_avg_mark["avg_final_mark"].min()
    max_mark = course_avg_mark["avg_final_mark"].max()
    
    if max_mark == min_mark:
        # Avoid division by zero if all courses have the same average mark.
        course_avg_mark["normalized_mark"] = 0
    else:
        course_avg_mark["normalized_mark"] = (course_avg_mark["avg_final_mark"] - min_mark) / (max_mark - min_mark)

    # 3. The difficulty is 1 minus the normalized mark.
    course_avg_mark["course_difficulty"] = 1 - course_avg_mark["normalized_mark"]
    
    return course_avg_mark[["course_id", "course_difficulty"]]

# Example usage with your labels
# Assuming you have a df_labels DataFrame with 'student_id', 'course_id', and 'final_mark'
# Note: You need to use your labels dataset for this, as it contains the final_mark.
# The `df_task_marks` you provided doesn't have the final_mark, but the labels dataset does.
# This revised function relies on the final outcome for a more accurate difficulty metric.
# Let's assume you have a df_labels DataFrame

course_difficulty = compute_difficulty_revised(df_labels)

# You can then merge this with your main training data on `course_id`.
df_train = df_train.merge(course_difficulty, on="course_id", how="left")

In [4]:
def get_past_performance(df_labels):
    """
    Computes a student's average past final mark and completion rate.
    """
    # Sort by date to process chronologically (assuming `final_mark` implies course completion)
    df_labels = df_labels.sort_values(by="course_id") # Assuming course_id is chronological

    student_data = []
    
    # Calculate past performance for each student and course
    for (student_id, course_id), group in df_labels.groupby(["student_id", "course_id"]):
        past_marks = df_labels[
            (df_labels["student_id"] == student_id) & (df_labels["course_id"] < course_id)
        ]["final_mark"]
        
        # Calculate average past mark
        avg_past_mark = past_marks.mean() if not past_marks.empty else 0
        
        # Calculate completion rate (assuming a mark > 0 is a completion)
        completion_rate = (past_marks > 0).mean() if not past_marks.empty else 0
        
        student_data.append({
            "student_id": student_id,
            "course_id": course_id,
            "average_past_final_mark": avg_past_mark,
            "completion_rate": completion_rate
        })
    
    return pd.DataFrame(student_data)
test = get_past_performance(df_labels)


In [5]:
test

,student_id,course_id,average_past_final_mark,completion_rate
0,STU001C1A,CRS873520,0.0,0.0
1,STU004FF9,CRS283DC7,0.0,0.0
2,STU004FF9,CRS7BD731,0.0,0.0
3,STU00641A,CRSCCEF31,0.0,0.0
4,STU0065A7,CRS675703,0.0,0.0
...,...,...,...,...
2955,STUFFDB9D,CRS94D1D4,0.0,0.0
2956,STUFFE655,CRS7A2125,0.0,0.0
2957,STUFFF26A,CRS9A7763,0.0,0.0
2958,STUFFF768,CRS6F5FBE,0.0,0.0


In [6]:
# Interaction feature
df_train["workload_x_course_difficulty"] = (
    df_train["non_graded_interactions_per_course"] * df_train["course_difficulty"]
)

# Merge past-performance features (from the previous cell's `test`).
df_train = df_train.merge(
    test,
    on=["student_id", "course_id"],
    how="left"
)
df_train[["average_past_final_mark", "completion_rate"]] = df_train[
    ["average_past_final_mark", "completion_rate"]
].fillna(0)

# Merge course ECTS.
ects_df = df_courses_tasks[['course_id', 'course_ects']].drop_duplicates()
df_train = df_train.merge(ects_df, on='course_id', how='left')

# Merge the target on keys (NOT by index — df_train and df_labels are not aligned).
df_train = df_train.merge(
    df_labels[['student_id', 'course_id', 'final_mark']],
    on=['student_id', 'course_id'],
    how='left'
)
df_train["final_mark_rounded"] = (df_train["final_mark"] / 10).round().astype("Int64") * 10

# Drop the raw mark so it doesn't leak into X, and drop rows with missing features/labels.
train_df = df_train.drop(columns=["final_mark"]).dropna().reset_index(drop=True)
train_df["final_mark_rounded"] = train_df["final_mark_rounded"].astype(int)


In [7]:
SNAPSHOT_DATE = pd.Timestamp("2024-01-01")  # reference for converting date columns → numeric years

def _preprocess_features(df: pd.DataFrame) -> pd.DataFrame:
    """Convert date-like object columns to numeric years from SNAPSHOT_DATE,
    then one-hot encode remaining object/category columns so sklearn can fit."""
    df = df.copy()

    # Date-like object cols → years (float).
    for col in df.select_dtypes(include=["object"]).columns:
        parsed = pd.to_datetime(df[col], errors="coerce")
        if parsed.notna().mean() >= 0.5:
            df[col] = (SNAPSHOT_DATE - parsed).dt.days / 365.25

    # One-hot encode any remaining string/categorical columns.
    obj_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    if obj_cols:
        df = pd.get_dummies(df, columns=obj_cols, drop_first=True)

    # Cast bools to ints so the whole frame is numeric.
    bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(int)

    return df


X = train_df.drop(columns=["student_id", "course_id", "final_mark_rounded"])
X = _preprocess_features(X)

y = train_df["final_mark_rounded"]

# Stratify only if every class has >=2 members (required by train_test_split).
class_counts = y.value_counts()
stratify_arg = y if (y.nunique() > 1 and class_counts.min() >= 2) else None

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_arg
)


C:\Users\Anurath\AppData\Local\Temp\ipykernel_27084\1798025843.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:
C:\Users\Anurath\AppData\Local\Temp\ipykernel_27084\1798025843.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[col], errors="coerce")
C:\Users\Anurath\AppData\Local\Temp\ipykernel_27084\1798025843.py:10: UserWarning: Could not infer fo

In [8]:
def optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val):
    """
    Optimizes classification models using Optuna. Returns weighted F1-score (to be maximized).
    """
    if model_name == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 100, 1000)
        max_depth = trial.suggest_int("max_depth", 3, 55)
        # sklearn requires min_samples_split >= 2
        min_samples_split = trial.suggest_int("min_samples_split", 2, 50)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 50)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    acc = accuracy_score(y_val, preds)
    f1 = f1_score(y_val, preds, average="weighted", zero_division=0)
    recall = recall_score(y_val, preds, average="weighted", zero_division=0)
    precision = precision_score(y_val, preds, average="weighted", zero_division=0)

    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("precision", precision)

    return f1


models_to_optimize = ["RandomForest"]
best_models = {}

for model_name in models_to_optimize:
    print(f"\n{'='*50}")
    print(f"Optimizing {model_name}...")
    print(f"{'='*50}")

    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda trial: optimize_model_cls(trial, model_name, X_train, y_train, X_val, y_val),
        n_trials=100,
        show_progress_bar=True,
    )

    best_models[model_name] = {
        "best_params": study.best_params,
        "best_f1": study.best_trial.user_attrs.get("f1"),
        "best_accuracy": study.best_trial.user_attrs.get("accuracy"),
        "best_recall": study.best_trial.user_attrs.get("recall"),
        "best_precision": study.best_trial.user_attrs.get("precision"),
    }

    print(f"\nBest F1-score for {model_name}: {best_models[model_name]['best_f1']:.4f}")
    print(f"Accuracy: {best_models[model_name]['best_accuracy']:.4f}")
    print(f"Recall: {best_models[model_name]['best_recall']:.4f}")
    print(f"Precision: {best_models[model_name]['best_precision']:.4f}")
    print(f"Best params: {study.best_params}")


[I 2026-05-26 22:04:01,804] A new study created in memory with name: no-name-1e704526-07ca-495e-b422-a8b2d9bd2230



Optimizing RandomForest...


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-26 22:04:03,914] Trial 0 finished with value: 0.19007799399820624 and parameters: {'n_estimators': 811, 'max_depth': 26, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.19007799399820624.
[I 2026-05-26 22:04:05,200] Trial 1 finished with value: 0.15264299571425863 and parameters: {'n_estimators': 465, 'max_depth': 11, 'min_samples_split': 41, 'min_samples_leaf': 43, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.19007799399820624.
[I 2026-05-26 22:04:07,321] Trial 2 finished with value: 0.15887124539211223 and parameters: {'n_estimators': 857, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 39, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.19007799399820624.
[I 2026-05-26 22:04:08,219] Trial 3 finished with value: 0.19847068568969983 and parameters: {'n_estimators': 286, 'max_depth': 32, 'min_samples_split': 13, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 3 with value: 0.19847